In [ ]:
import pandas as pd

df = pd.read_csv("./ia_tweets.csv")

df.head()

In [ ]:
df.dtypes

In [ ]:
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0

df['idioma'] = df['text'].apply(detect)

df = df[df["idioma"] == "es"]

df = df.drop(columns=["ID","user_has_extended_profile", "user_is_translator", "user_protected", "idioma"])

In [ ]:
def filtrar_por_polaridad(field):
    return df[df["polarity"] == field]

cant_negativos = len(filtrar_por_polaridad("N"))
cant_neutros = len(filtrar_por_polaridad("NEU"))
cant_positivos =  len(filtrar_por_polaridad("P"))

print('Cantidad de tweets: ', len(df))
print('Tweets Negativos: ', cant_negativos, ', Neutros' , cant_neutros,', Positivos', cant_positivos)

# Cantidad de datos desbalanceada, se tiene mucho mas datos neutros que negativos o positivos.

In [ ]:
cantidad_na = df.isna().sum()
print(cantidad_na)

#(cantidad_negativos_columna("favorite_count"))

# No hay nulls o valores

In [ ]:
cantidad_negativos = (df.select_dtypes(include='number') < 0).sum()
print(cantidad_negativos)

In [ ]:
import matplotlib.pyplot as plt

# Datos
clases = ['Positivo', 'Negativo', 'Neutral']
cantidades = [cant_positivos, cant_negativos, cant_neutros]

# Crear gráfico
plt.pie(cantidades, labels=clases, autopct='%1.1f%%')
plt.title('Distribución de sentimientos')
plt.show()

In [ ]:
print(df["favorite_count"].mean())
print(df["favorite_count"].max())
print(df["favorite_count"].min())

plt.scatter(range(0,len(df)), df['favorite_count'])

plt.xlabel('Ids')
plt.ylabel('Cantidad de me gusta')
plt.title('ID')

plt.show()

In [ ]:
print(df["retweet_count"].mean())
print(df["retweet_count"].max())
print(df["retweet_count"].min())

plt.scatter(range(0,len(df)), df['retweet_count'])

plt.xlabel('Id')
plt.ylabel('Cantidad de retweets')
plt.title('Retweets')

plt.show()

In [ ]:
df['tweet_interactions'] = df['favorite_count'] + df['retweet_count']

tweets_con_interaccion = (df['tweet_interactions'] > 0).sum()
tweets_sin_interaccion = (df['tweet_interactions'] == 0).sum()

positivos_con_interaccion = (df[df['polarity'] == 'P']['tweet_interactions'] > 1).sum()
neutros_con_interaccion = (df[df['polarity'] == 'NEU']['tweet_interactions'] > 1).sum()
negativos_con_interaccion = (df[df['polarity'] == 'N']['tweet_interactions'] > 1).sum()
interaccion_positivos_media = df[df['polarity'] == 'P']['tweet_interactions'].mean()
interaccion_negativos_media = df[df['polarity'] == 'N']['tweet_interactions'].mean()

print("Cantidad de tweets positivo con interaccion: ", positivos_con_interaccion)
print("Cantidad de tweets negativos con interaccion: ", negativos_con_interaccion)
print("Media de interacciones en positivos: ", interaccion_positivos_media)
print("Media de interacciones en negativos: ", interaccion_negativos_media)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].pie([positivos_con_interaccion, neutros_con_interaccion, negativos_con_interaccion], labels=["positivos", "neutros", "negativos"], autopct='%1.1f%%')
ax[0].set_title('Porcentaje de polaridades con interaccion')

ax[1].pie([tweets_con_interaccion, tweets_sin_interaccion],
  labels=["Interactuados", "No interactudos"], autopct='%1.1f%%')
ax[1].set_title('Porcentaje de tweets con interaccion')

plt.tight_layout()
plt.show()

In [ ]:
 #text
#polarity
#favorite_count
#retweet_count
#user_followers_count
#user_friends_count
#user_favourites_count         TEST (Likes que dio)
#user_statuses_count           TEST (Publicaciones y comentarios suyos)
#user_verified                 TEST
#user_default_profile          TEST (Pagina de perfil default)

import unicodedata
import emoji
from emosent import get_emoji_sentiment_rank

def emoji_features(text):
    scores = []
    for e in emoji.emoji_list(str(text)):
        try:
            rank = get_emoji_sentiment_rank(e["emoji"])
        except KeyError:
            rank = None
        if rank is not None:
            scores.append(rank["sentiment_score"])
    if not scores:
        return 0.0, 0, 0
    return (
        sum(scores) / len(scores),
        sum(s > 0.1 for s in scores),
        sum(s < -0.1 for s in scores),
    )

def contar_emojis(texto):
    return sum(1 for caracter in str(texto) if unicodedata.category(caracter) == 'So')

# Cálculo de características de emojis
df[["emoji_score", "emoji_pos", "emoji_neg"]] = df["text"].apply(
    lambda t: pd.Series(emoji_features(t))
)

df["length"] = df['text'].apply(lambda texto: len(texto.split()))
df["hashtag_count"] = df['text'].apply(lambda texto: texto.count('#'))
df["mention_count"] = df['text'].apply(lambda texto: texto.count('@'))
df["url_count"] = df['text'].apply(lambda texto: texto.count('http'))
df["punctuation_count"] = df['text'].apply(lambda texto: texto.count('?') + texto.count('!'))
df['emoji_count'] = df['text'].apply(contar_emojis)

# Listado de columnas numéricas y de emojis combinadas
num_cols = [
    "favorite_count", "retweet_count", "user_followers_count",
    "user_friends_count", "user_favourites_count", "user_statuses_count",
    "tweet_interactions", "hashtag_count", "mention_count",
    "punctuation_count", "emoji_count", "emoji_score", "emoji_pos", "emoji_neg"
]

# :url: :mencion:

#Campos extra
#Longitud (palabras)
#Cantidad de hashtags
#Cantidad de usuarios mencionados
#Cantidad de emojis
#Cantidad de urls
#Cantidad de signos de exclamacion/interrogacion

In [ ]:
df["text"] = df["text"].str.replace(r"http\S+|www\.\S+", "URL", regex=True)
df["text"] = df["text"].str.replace(r"@\w+", "MENCION", regex=True)

df.head()

In [ ]:
from sklearn.model_selection import train_test_split

# Pasamos tanto la columna text como las numéricas/emojis al split (80% Train, 20% Test)
X = df[["text"] + num_cols]
y = df["polarity"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Proporción de Train sobre el total:", len(X_train)/len(df))

In [ ]:
from sklearn.preprocessing import StandardScaler

number_columns = ["favorite_count", "retweet_count", "user_followers_count", "user_friends_count",
                  "user_favourites_count", "user_statuses_count", "tweet_interactions", "hashtag_count",
                  "mention_count"]

# Solo para visualización en el notebook si deseas inspeccionar los datos escalados
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_train_scaled[number_columns] = scaler.fit_transform(X_train[number_columns])

X_train_scaled.head()
y_train.head()

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
import string

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
stop_words = set(stopwords.words('spanish'))

class TextCleaner(BaseEstimator, TransformerMixin):
    def __init__(self, method=None):
        self.method = method

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        stop_words = set(stopwords.words('spanish'))
        punctuation = set(string.punctuation)
        stemmer = PorterStemmer()
        lemmatizer = WordNetLemmatizer()

        clean_X = []
        X_list = X.values.ravel() if isinstance(X, (pd.DataFrame, pd.Series)) else X

        for text in X_list:
            tokens = nltk.word_tokenize(str(text).lower())
            tokens = [t for t in tokens if t not in stop_words and t not in punctuation]

            if self.method == 'stem':
                tokens = [stemmer.stem(t) for t in tokens]
            elif self.method == 'lemma':
                tokens = [lemmatizer.lemmatize(t) for t in tokens]

            clean_X.append(" ".join(tokens))
        return clean_X

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

cleaner = TextCleaner("lemma")

vectores = cleaner.transform(df["text"])
vectorizer_BoW = CountVectorizer()
vectorizer_Tfidf = TfidfVectorizer()

matriz_BoW = vectorizer_BoW.fit_transform(vectores)
matriz_Tfidf = vectorizer_Tfidf.fit_transform(vectores)

print(vectorizer_BoW.get_feature_names_out())
print(matriz_BoW.toarray()[0])
#print(vectores_Tfidf.shape)

In [ ]:
from sentence_transformers import SentenceTransformer

class EmbeddingTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, model_name='hiiamsid/sentence_similarity_spanish_es'):
        self.model_name = model_name
        self.model = None

    def fit(self, X, y=None):
        if self.model is None:
            self.model = SentenceTransformer(self.model_name)
        return self

    def transform(self, X):
        X_list = X.values.ravel().tolist() if isinstance(X, (pd.DataFrame, pd.Series)) else list(X)
        return self.model.encode(X_list, show_progress_bar=False)

embedding = EmbeddingTransformer()

In [ ]:
embedding.fit(df["text"])
text = embedding.transform(df["text"])
print(text.shape)
print(text)

In [ ]:
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.base import BaseEstimator, TransformerMixin

class DenseTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if hasattr(X, "toarray"):
            return X.toarray()
        return X

# Pipeline para la rama de texto
text_pipeline = Pipeline([
    ('cleaner', TextCleaner()),
    ('vectorizer', CountVectorizer()),
    ('to_dense', DenseTransformer())
])

# Integración con ColumnTransformer
preprocessor = ColumnTransformer([
    ('text', text_pipeline, 'text'),
    ('num', StandardScaler(), num_cols)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GaussianNB())
])

# Param grid
param_grid = [
    {
        'preprocessor__num': [StandardScaler(), RobustScaler()],
        'preprocessor__text__cleaner__method': [None, 'stem', 'lemma'],
        'preprocessor__text__vectorizer': [TfidfVectorizer()],
        'preprocessor__text__vectorizer__ngram_range': [(1, 1), (1, 2)],
        'preprocessor__text__vectorizer__min_df': [1, 2],
        'preprocessor__text__vectorizer__max_df': [0.85, 1.0],
        'preprocessor__text__vectorizer__sublinear_tf': [True, False],
        'classifier__var_smoothing': np.logspace(-9, -1, 5)
    },
    {
        'preprocessor__num': [StandardScaler(), RobustScaler()],
        'preprocessor__text__cleaner__method': [None, 'stem', 'lemma'],
        'preprocessor__text__vectorizer': [CountVectorizer()],
        'preprocessor__text__vectorizer__ngram_range': [(1, 1), (1, 2)],
        'preprocessor__text__vectorizer__min_df': [1, 2],
        'preprocessor__text__vectorizer__max_df': [0.85, 1.0],
        'classifier__var_smoothing': np.logspace(-9, -1, 5)
    }
]

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring='roc_auc_ovr',
    n_jobs=1,
    verbose=1
)
grid_search.fit(X_train, y_train)

In [ ]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import classification_report, roc_auc_score

# Aplicar la limpieza de texto
cleaner = TextCleaner()
X_train_clean_text = cleaner.transform(X_train["text"])
X_test_clean_text = cleaner.transform(X_test["text"])

# Crear DataFrames auxiliares para mantener columnas alineadas
X_train_proc = pd.DataFrame({"text": X_train_clean_text}, index=X_train.index)
X_test_proc = pd.DataFrame({"text": X_test_clean_text}, index=X_test.index)

for col in num_cols:
    X_train_proc[col] = X_train[col].values
    X_test_proc[col] = X_test[col].values

# Definir Preprocesador y Pipeline
preprocessor = ColumnTransformer([
    ("text", TfidfVectorizer(), "text"),
    ("num", StandardScaler(), num_cols)
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("to_dense", DenseTransformer()),
    ("classifier", GaussianNB())
])

# Param Grid
param_grid = [
    # TF-IDF
    {
        "preprocessor__num": [StandardScaler(), RobustScaler()],
        "preprocessor__text": [TfidfVectorizer()],
        "preprocessor__text__strip_accents": [None, "unicode"],
        "preprocessor__text__ngram_range": [(1, 1), (1, 2)],
        "preprocessor__text__min_df": [1, 2, 5],
        "preprocessor__text__max_df": [0.7, 0.85, 1.0],
        "preprocessor__text__norm": ["l1", "l2"],
        "preprocessor__text__use_idf": [True, False],
        "preprocessor__text__sublinear_tf": [True, False],
        "classifier__var_smoothing": np.logspace(-9, -1, 15),
    },
    # CountVectorizer (BoW)
    {
        "preprocessor__num": [StandardScaler(), RobustScaler()],
        "preprocessor__text": [CountVectorizer()],
        "preprocessor__text__strip_accents": [None, "unicode"],
        "preprocessor__text__ngram_range": [(1, 1), (1, 2)],
        "preprocessor__text__min_df": [1, 2, 5],
        "preprocessor__text__max_df": [0.7, 0.85, 1.0],
        "preprocessor__text__binary": [True, False],
        "classifier__var_smoothing": np.logspace(-9, -1, 15),
    },
]

# RandomizedSearchCV
grid_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=300,
    cv=3,
    scoring={
        "f1_macro": "f1_macro",
        "roc_auc_ovr": "roc_auc_ovr",
    },
    refit="roc_auc_ovr",
    random_state=42,
    n_jobs=1,
    verbose=1,
)

grid_search.fit(X_train_proc, y_train)

idx = grid_search.best_index_

print("\n--- RESULTADOS TF-IDF / BoW ---")
print("Mejores parámetros:", grid_search.best_params_)
print("Mejor F1 macro (CV):", grid_search.cv_results_["mean_test_f1_macro"][idx])
print("Mejor ROC-AUC OvR (CV):", grid_search.best_score_)

# Evaluación final
proba = grid_search.predict_proba(X_test_proc)
print("ROC-AUC test (TF-IDF):", roc_auc_score(y_test, proba, multi_class="ovr"))
print(classification_report(y_test, grid_search.predict(X_test_proc)))

In [ ]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score

# Pre-calcular Embeddings de texto una sola vez (rápido)
print("Generando embeddings semánticos para Train y Test...")
embedding = EmbeddingTransformer()
emb_train = embedding.fit_transform(X_train['text'])
emb_test = embedding.transform(X_test['text'])

# Extraer y preparar columnas numéricas
num_train = X_train[num_cols].values
num_test = X_test[num_cols].values

# Concatenar horizontalmente: primeras 384 columnas = embeddings, resto = num_cols
X_train_emb_full = np.hstack([emb_train, num_train])
X_test_emb_full = np.hstack([emb_test, num_test])

n_emb_dim = emb_train.shape[1]
n_total_dim = X_train_emb_full.shape[1]

# PCA a la sección de Embeddings y Escalado a la sección Numérica
preprocessor_emb = ColumnTransformer([
    ('text_pca', PCA(), slice(0, n_emb_dim)),
    ('num_scaler', StandardScaler(), slice(n_emb_dim, n_total_dim))
])

pipe = Pipeline([
    ('preprocessor', preprocessor_emb),
    ('gnb', GaussianNB())
])

# Encoding de etiquetas
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# Param Grid
param_grid_emb = {
    "preprocessor__num_scaler": [StandardScaler(), RobustScaler()],
    "preprocessor__text_pca__n_components": [5, 10, 15, 20, 30, 40, 50, 75, 100, 150, 200],
    "preprocessor__text_pca__whiten": [True, False],
    "preprocessor__text_pca__svd_solver": ["auto", "full", "randomized"],
    "gnb__var_smoothing": np.logspace(-9, -1, 30),
}

scoring = {
    "f1_macro": "f1_macro",
    "roc_auc_ovr": "roc_auc_ovr"
}

grid_emb = RandomizedSearchCV(
    pipe,
    param_distributions=param_grid_emb,
    n_iter=500,
    cv=3,
    scoring=scoring,
    refit="roc_auc_ovr",
    random_state=42,
    n_jobs=1,
    verbose=1,
)

print("Iniciando búsqueda aleatoria de hiperparámetros...")
grid_emb.fit(X_train_emb_full, y_train_encoded)

idx = grid_emb.best_index_
print("\n--- RESULTADOS EMBEDDINGS ---")
print("Mejores parámetros:", grid_emb.best_params_)
print("F1 macro (CV):", grid_emb.cv_results_["mean_test_f1_macro"][idx])
print("ROC-AUC (CV):", grid_emb.best_score_)

# Evaluación final sobre el conjunto de Test
proba_emb = grid_emb.predict_proba(X_test_emb_full)
auc_score = roc_auc_score(y_test_encoded, proba_emb, multi_class="ovr")

print("ROC-AUC test (embeddings):", auc_score)
print(classification_report(y_test_encoded, grid_emb.predict(X_test_emb_full), target_names=le.classes_))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Matriz de Confusión para Modelo TF-IDF
y_pred_tfidf = grid_search.predict(X_test_proc)

fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_tfidf,
    display_labels=grid_search.classes_,
    cmap=plt.cm.Blues,
    ax=ax
)
ax.set_title("Matriz de Confusión - TF-IDF + Naive Bayes")
plt.grid(False)
plt.show()

# Matriz de Confusión para Modelo Embeddings + PCA
y_pred_emb = grid_emb.predict(X_test_emb_full)

fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test_encoded,
    y_pred_emb,
    display_labels=le.classes_,
    cmap=plt.cm.Greens,
    ax=ax
)
ax.set_title("Matriz de Confusión - Embeddings + PCA + Naive Bayes")
plt.grid(False)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Gráfica de Complejidad para Embeddings
results_emb_df = pd.DataFrame(grid_emb.cv_results_)

# Extraer y ordenar por var_smoothing
results_emb_df = results_emb_df.sort_values(by='param_gnb__var_smoothing')

var_smoothing_emb = results_emb_df['param_gnb__var_smoothing']
scores_auc_emb = results_emb_df['mean_test_roc_auc_ovr']
scores_f1_emb = results_emb_df['mean_test_f1_macro']

plt.figure(figsize=(9, 5))
plt.plot(var_smoothing_emb, scores_auc_emb, marker='o', label='ROC-AUC (CV)', color='b')
plt.plot(var_smoothing_emb, scores_f1_emb, marker='s', label='F1-Macro (CV)', color='g')

plt.xscale('log')
plt.xlabel('Parámetro de suavizado (var_smoothing)')
plt.ylabel('Puntaje (Score)')
plt.title('Gráfica de Complejidad - GaussianNB (Embeddings)')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.show()

# Gráfica de Complejidad para TF-IDF / BoW
results_df = pd.DataFrame(grid_search.cv_results_)

# Obtengo las mejores configuraciones encontradas para el vectorizador
best_ngram = grid_search.best_params_['preprocessor__text__ngram_range']
best_vect = grid_search.best_params_['preprocessor__text']

# Filtro los resultados probados
subset = results_df[
    results_df['param_preprocessor__text__ngram_range'] == best_ngram
    ].copy()

# Ordeno var_smoothing para una visualización correcta
subset = subset.sort_values(by='param_classifier__var_smoothing')

plt.figure(figsize=(9, 5))
plt.plot(
    subset['param_classifier__var_smoothing'],
    subset['mean_test_roc_auc_ovr'],
    marker='o',
    color='r',
    label='ROC-AUC OvR (CV)'
)
plt.plot(
    subset['param_classifier__var_smoothing'],
    subset['mean_test_f1_macro'],
    marker='s',
    color='orange',
    label='F1-Macro (CV)'
)

plt.xscale('log')
plt.xlabel('Parámetro de suavizado (var_smoothing)')
plt.ylabel('Puntaje (Score)')
plt.title(f'Gráfica de Complejidad TF-IDF / BoW (N-grams: {best_ngram})')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize


def graficar_curva_roc_multiclase(y_true, probas, clases, titulo="Curva ROC Multiclase"):
    # 1. Binarizar las etiquetas reales utilizando el listado de clases original
    y_true_bin = label_binarize(y_true, classes=clases)
    n_classes = len(clases)

    plt.figure(figsize=(8, 6))

    # 2. Calcular ROC y AUC para cada clase (N, NEU, P)
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], probas[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(
            fpr, tpr,
            lw=2,
            label=f'Clase {clases[i]} (AUC = {roc_auc:.3f})'
        )

    # Diagonal de referencia de clasificación al azar
    plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Azar (AUC = 0.500)')

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Tasa de Falsos Positivos (FPR)')
    plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
    plt.title(titulo)
    plt.legend(loc="lower right")
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.show()

# Curva ROC para el modelo TF-IDF / BoW
proba_tfidf = grid_search.predict_proba(X_test_proc)
clases_tfidf = grid_search.classes_  # Array ['N', 'NEU', 'P']

graficar_curva_roc_multiclase(
    y_test,
    proba_tfidf,
    clases_tfidf,
    titulo="Curva ROC (One-vs-Rest) - TF-IDF + Naive Bayes"
)

# Curva ROC para el modelo Embeddings + PCA
proba_emb = grid_emb.predict_proba(X_test_emb_full)

graficar_curva_roc_multiclase(
    y_test,
    proba_emb,
    le.classes_,  # Muestra correctamente 'N', 'NEU', 'P' en la leyenda
    titulo="Curva ROC (One-vs-Rest) - Embeddings + PCA + Naive Bayes"
)

In [ ]:
import pandas as pd
import numpy as np


def predecir_frase(texto_nuevo, num_dict=None):
    # Si no se pasan métricas numéricas/emojis, se calculan dinámicamente
    if num_dict is None:
        emoji_score, emoji_pos, emoji_neg = emoji_features(texto_nuevo)
        num_dict = {
            "favorite_count": 0,
            "retweet_count": 0,
            "user_followers_count": 100,
            "user_friends_count": 100,
            "user_favourites_count": 100,
            "user_statuses_count": 500,
            "tweet_interactions": 0,
            "hashtag_count": texto_nuevo.count('#'),
            "mention_count": texto_nuevo.count('@'),
            "punctuation_count": texto_nuevo.count('?') + texto_nuevo.count('!'),
            "emoji_count": contar_emojis(texto_nuevo),
            "emoji_score": emoji_score,
            "emoji_pos": emoji_pos,
            "emoji_neg": emoji_neg
        }

    # -------------------------------------------------------------------------
    # MODELO 1: TF-IDF / BoW
    # -------------------------------------------------------------------------
    clean_text = TextCleaner().transform([texto_nuevo])[0]
    df_tfidf_in = pd.DataFrame([{"text": clean_text, **num_dict}])

    pred_tfidf = grid_search.predict(df_tfidf_in)[0]
    probas_tfidf = grid_search.predict_proba(df_tfidf_in)[0]

    # -------------------------------------------------------------------------
    # MODELO 2: Embeddings + PCA
    # -------------------------------------------------------------------------
    emb_vector = embedding.transform([texto_nuevo])

    num_vector = np.array([[num_dict[col] for col in num_cols]])
    X_emb_in = np.hstack([emb_vector, num_vector])

    pred_emb_num = grid_emb.predict(X_emb_in)[0]
    pred_emb = le.inverse_transform([pred_emb_num])[0]
    probas_emb = grid_emb.predict_proba(X_emb_in)[0]

    # -------------------------------------------------------------------------
    # IMPRESIÓN DE RESULTADOS
    # -------------------------------------------------------------------------
    print("=" * 60)
    print(f"Frase a evaluar: '{texto_nuevo}'")
    print("=" * 60)

    print("\n--- MODELO TF-IDF / BoW ---")
    print(f"Predicción: {pred_tfidf}")
    for clase, prob in zip(grid_search.classes_, probas_tfidf):
        print(f"  Probabilidad [{clase}]: {prob:.4f}")

    print("\n--- MODELO EMBEDDINGS + PCA ---")
    print(f"Predicción: {pred_emb}")
    for clase, prob in zip(le.classes_, probas_emb):
        print(f"  Probabilidad [{clase}]: {prob:.4f}")
    print("=" * 60)

# =============================================================================
# PRUEBAS DE EJEMPLO
# =============================================================================

predecir_frase("Me encanta la IA, responde todas mis preguntas al instante!")
predecir_frase("La verdad, horrible pensar hasta dónde puede llegar la inteligencia artificial. Miedo.")
predecir_frase("No sé que pensar, me genera dudas.")
predecir_frase("Tengo una nueva moto.")
predecir_frase("Buenísimo lo de la IA, che. Quién hubiera pensado que iban a pasar estas cosas, no?")